In [2]:
# Install dependencies
%pip install -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# Import necessary libraries
import pandas as pd

In [4]:
df_limpio = pd.read_csv('../data/raw/movilidad_sensores_LIMPIO.csv')
df_limpio.info()

<class 'pandas.DataFrame'>
RangeIndex: 1440 entries, 0 to 1439
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   sensor_id         1440 non-null   str    
 1   ubicacion         1440 non-null   str    
 2   tipo_via          1440 non-null   str    
 3   timestamp         1440 non-null   str    
 4   conteo_vehiculos  1440 non-null   int64  
 5   temperatura_c     1440 non-null   float64
 6   condicion_clima   1440 non-null   str    
 7   lat               1440 non-null   float64
 8   lon               1440 non-null   float64
dtypes: float64(3), int64(1), str(5)
memory usage: 168.9 KB


## Inventario de variables - Dataset limpio

| Variable | Tipo de dato | Formato de origen | Fuente |
|---|---|---|---|
| `sensor_id` | Nominal | CSV | Estructurada |
| `ubicacion` | Nominal | CSV | Estructurada |
| `tipo_via` | Nominal | CSV | Estructurada |
| `timestamp` | Fecha | CSV | Estructurada |
| `conteo_vehiculos` | Discreto | CSV | Estructurada |
| `temperatura_c` | Continuo | CSV | Estructurada |
| `condicion_clima` | Nominal | CSV | Estructurada |
| `lat` | Geoespacial | CSV | Estructurada |
| `lon` | Geoespacial | CSV | Estructurada |

In [5]:
df_contaminado = pd.read_csv('../data/raw/movilidad_sensores_CONTAMINADO.csv')
df_contaminado.info()

<class 'pandas.DataFrame'>
RangeIndex: 1455 entries, 0 to 1454
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   sensor_id         1455 non-null   str    
 1   ubicacion         1455 non-null   str    
 2   tipo_via          1455 non-null   str    
 3   timestamp         1455 non-null   str    
 4   conteo_vehiculos  1310 non-null   float64
 5   temperatura_c     1383 non-null   float64
 6   condicion_clima   1368 non-null   str    
 7   lat               1455 non-null   float64
 8   lon               1455 non-null   float64
dtypes: float64(4), str(5)
memory usage: 170.2 KB


## Inventario de variables - Dataset contaminado

| Variable | Tipo de dato | Formato de origen | Fuente |
|---|---|---|---|
| `sensor_id` | Nominal | CSV | Estructurada |
| `ubicacion` | Nominal | CSV | Estructurada |
| `tipo_via` | Nominal | CSV | Estructurada |
| `timestamp` | Fecha | CSV | Estructurada |
| `conteo_vehiculos` | Discreto | CSV | Estructurada |
| `temperatura_c` | Continuo | CSV | Estructurada |
| `condicion_clima` | Nominal | CSV | Estructurada |
| `lat` | Geoespacial | CSV | Estructurada |
| `lon` | Geoespacial | CSV | Estructurada |

In [6]:
df_json = pd.read_json('../data/raw/clima_api_log.json')
df_json_flat = pd.concat(
    [
        df_json.drop(columns=['location', 'weather']).reset_index(drop=True),
        pd.json_normalize(df_json['location']),
        pd.json_normalize(df_json['weather'])
    ],
    axis=1
)
df_json_flat.info()

<class 'pandas.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   request_id    120 non-null    str           
 1   timestamp     120 non-null    datetime64[us]
 2   sensor_id     120 non-null    str           
 3   lat           120 non-null    float64       
 4   lon           120 non-null    float64       
 5   temp_c        116 non-null    float64       
 6   condition     120 non-null    str           
 7   humidity_pct  120 non-null    int64         
dtypes: datetime64[us](1), float64(3), int64(1), str(3)
memory usage: 9.6 KB


## Inventario de variables - Dataset JSON

| Variable | Tipo de dato | Formato de origen | Fuente |
|---|---|---|---|
| `request_id` | Nominal | JSON anidado | Semi-estructurada |
| `timestamp` | Fecha | JSON anidado | Semi-estructurada |
| `sensor_id` | Nominal | JSON anidado | Semi-estructurada |
| `lat` | Geoespacial | JSON anidado | Semi-estructurada |
| `lon` | Geoespacial | JSON anidado | Semi-estructurada |
| `temp_c` | Continuo | JSON anidado | Semi-estructurada |
| `condition` | Nominal | JSON anidado | Semi-estructurada |
| `humidity_pct` | Continuo | JSON anidado | Semi-estructurada |

## Reflexión

**¿Qué información se pierde o se distorsiona al forzar una fuente no estructurada/semi-estructurada dentro de una tabla rectangular?**

Se pueden perder la estandarización de los tipos de datos. La relación y jerarquía entre los campos, los cuales pueden brindar información adicional sobre la estructura de los datos. Además, al venir de un formato no estructurado, los datos pueden tener campos opcionales, lo que puede llevar a la pérdida de información si se ignoran o se rellenan con valores nulos.

## Tarea 2 — Diagnóstico GIGO

Functiones utilizadas en la etapa de pre-procesamiento para encontrar problemas en los datos

In [ ]:
# Functions used to identify the data quality issues in the dataset

print("Null values per column:")
print(df_contaminado.isnull().sum())

print("\nUnique values in 'condicion_clima':")
print(df_contaminado["condicion_clima"].unique().value_counts())

print("\nNumber of duplicated rows based on 'sensor_id' and 'timestamp':")
print(df_contaminado.duplicated(subset=["sensor_id","timestamp"]).sum())

print("\nInvalid timestamps:")
print(pd.to_datetime(df_contaminado["timestamp"], errors="coerce").isna().sum())

print("\nNegative values in 'conteo_vehiculos':")
print((df_contaminado["conteo_vehiculos"] < 0).sum())

display(df_contaminado["lon"].describe())
print("\nLatitude and Longitude values outside Medellín's expected range ")
print((~df_contaminado["lat"].between(6.09, 6.39)).sum() & (~df_contaminado["lon"].between(-75.7, -75.47)).sum())

print("\nPlaceholder values (999) in 'conteo_vehiculos':")
print((df_contaminado["conteo_vehiculos"] >= 999).sum())


Null values per column:
sensor_id             0
ubicacion             0
tipo_via              0
timestamp             0
conteo_vehiculos    145
temperatura_c        72
condicion_clima      87
lat                   0
lon                   0
dtype: int64

Unique values in 'condicion_clima':
Soleado     1
Nublado     1
Sol         1
Lluvia      1
Nubes       1
soleado     1
LLUVIA      1
SOLEADO     1
lluvia      1
nublado     1
lluvioso    1
Name: count, dtype: int64

Number of duplicated rows based on 'sensor_id' and 'timestamp':
14

Invalid timestamps:
90

Negative values in 'conteo_vehiculos':
6


count    1455.000000
mean      -68.768157
std        22.612127
min       -75.591600
25%       -75.585105
50%       -75.572430
75%       -75.565360
max         6.287500
Name: lon, dtype: float64


Latitude values outside Medellín's expected range (6.02–6.25):
121

Placeholder values (999) in 'conteo_vehiculos':
4


Hicimos un análisis exploratorio sobre el dataset que escogimos que es el de **Tráfico** y encontramos varios problemas que pueden afectar totalmente a la hora de analizar y mirar los resultados arrojados.

| Problema | Columna(s) | Cómo lo detectamos | Por qué nos importa |
|---|---|---|---|
| Datos nulos: son 145, 72 y 87 respectivamente | `conteo_vehiculos`, `temperatura_c`, `condicion_clima` | `df.isnull().sum()` | El conteo es algo fundamental en esta fuente de datos, si faltan lecturas, un pico real puede pasar desapercibido o podría alterar el resultado final. |
| El clima está escrito de 11 formas para 3 valores reales (Sol, soleado, SOLEADO...) | `condicion_clima` | `df["condicion_clima"].value_counts(dropna=False)` | Si el clima queda dividido en etiquetas que son lo mismo pero escrito de forma diferente, tendremos la información segmentada, con valores asociados que no son precisos. |
| 14 entradas tienen el mismo sensor y timestamp | `sensor_id`, `timestamp` | `df.duplicated(subset=["sensor_id","timestamp"]).sum()` | Un mismo sensor en la misma unidad de tiempo tiene varias lecturas y esto no tiene sentido en el mundo real. Por ende, lo que nos proporciona es información inconsistente. No se hace un análisis completamente duplicadas, ya que son un subconjunto de los resultados de este análisis.
| Fechas en cinco formatos distintos y 90 de ellas no se pueden convertir | `timestamp` | `pd.to_datetime(df["timestamp"], errors="coerce").isna().sum()` | Si no tenemos el timestamp bien definido, no mediremos correctamente los datos en términos de horas pico. Por ende, se deberían convertir las que sean posibles, y el resto se eliminarían porque no aportan información relevante.  |
| 6 conteos vehiculares por debajo de 0 | `conteo_vehiculos` | `(df["conteo_vehiculos"] < 0).sum()` | Un conteo de vehículos no puede ser negativo. Esto es una falla del sensor que distorsiona cualquier medida. |
| 242 filas de lat fuera del rango de Medellín (6.02–6.25) | `lat` | `(~df["lat"].between(6.02, 6.25)).sum()` | Al comparar contra el rango de coordenadas de Medellín, esas entradas quedan por fuera y ubicarían el sensor en un lugar equivocado. |
| Valores centinela: 4 lecturas con 99999 (`>= 999`) | `conteo_vehiculos` | `(df["conteo_vehiculos"] >= 999).sum()` | El 99999 dispara la media de 19 a 328, haciendo que los datos no reflejen la realidad del tráfico. |

# Tarea 3 - Transformación y limpieza con pandas

Corregimos los problemas utilizando la siguiente estructura: Primero quitamos los duplicados de filas que son duplicados exactos, luego nos encargamos de manejar el tema de que las fechas estaban duplicadas y que por consecuencia los eventos también. Despues nos encargamos de manejar el tema de valores imposibles y las coordenadas de Medellín que estaban desbordadas, para finalmente hacer la imputación de valores nulos cuando ya tenemos el resto del dataset estable y confiable.

In [10]:
df_contaminado_copy = df_contaminado.copy() #create a copy of the original to not lose the original data

# Duplicados exactos

Encontramos 7 filas que son iguales en todas las columnas, es decir que aportan información repetida en el mismo instante de tiempo, haciendo que se infle el conteo que tiene el sensor. Por esta razón, decidimos eliminarlas, ya que no hay pérdida real.

In [ ]:
df_contaminado_copy = df_contaminado_copy.drop_duplicates()

# Diferentes tipos de fecha mezcladas

Usamos datetime para unificar todas las fechas que sean posibles. Sin embargo, hay 90 entradas que no se pueden convertir dado que tienen formatos sueltos, texto en español o valores que no tienen sentido. Por esta razón, decidimos eliminar esas entradas ya que afectarían los resultados analizados sobre la franja horaria. Las 90 entradas son más o menos el 6% de los datos, por lo tanto decidimos asumir la pérdida que correr el riesgo que nos puede dar sobre los resultados.

In [ ]:
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
df = df.dropna(subset=["timestamp"])